# SatDiff — Kaggle training

Kaggle gives 30 GPU-hours/week, guaranteed. Colab guarantees nothing — that is
why this notebook exists.

## One-time setup

1. **Phone-verify the account** — kaggle.com/settings. GPU *and* Internet are both
   locked behind it.
2. Right sidebar: **Accelerator → GPU T4 x2**. Not P100 — the T4 has fp16 tensor
   cores and this config trains in mixed precision.
3. Right sidebar: **Internet → On**.
4. **Add-ons → Secrets** → add `HF_TOKEN`, a *write* token from
   hf.co/settings/tokens.

## The thing that bites

`/kaggle/working` is wiped between sessions and there is no Drive to fall back
on. The Hub is the only durable store, so `HF_TOKEN` is not optional — without
it a dead session costs you the whole run.

Every cell sets `PYTHONPATH` and `cd`s for itself. Changing the accelerator
restarts the kernel and drops that state, and a cell that assumed an earlier
cell had set it fails with a confusing `No module named 'satdiff'`.

In [8]:
# 1. GPU check. Anything other than True here and nothing below will work.
import torch
print("cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "Sidebar -> Accelerator -> GPU T4 x2")

cuda: True
Tesla T4


In [12]:
%cd /kaggle/working
!ls -la /kaggle/working/
!git clone -q https://github.com/DrKingSchultz69/satdiff-v2.git /kaggle/working/satdiff-v2
!mv /kaggle/working/ckpt_backup /kaggle/working/satdiff-v2/checkpoints
!ls -la /kaggle/working/satdiff-v2/checkpoints/

/kaggle/working
total 16
drwxr-xr-x 4 root root 4096 Aug  8 18:44 .
drwxr-xr-x 5 root root 4096 Aug  8 18:19 ..
drwxr-xr-x 3 root root 4096 Aug  8 18:44 ckpt_backup
drwxr-xr-x 2 root root 4096 Aug  8 18:19 .virtual_documents
total 1075404
drwxr-xr-x 3 root root       4096 Aug  8 18:44 .
drwxr-xr-x 9 root root       4096 Aug  8 18:45 ..
drwxr-xr-x 2 root root       4096 Aug  8 18:44 checkpoints
-rw-r--r-- 1 root root 1101195745 Aug  7 22:31 last.pt


In [11]:
!mkdir -p /kaggle/working/satdiff-v2/checkpoints
!mv /kaggle/working/satdiff-v2/checkpoints /kaggle/working/ckpt_backup && rm -rf /kaggle/working/satdiff-v2
!git clone -q https://github.com/DrKingSchultz69/satdiff-v2.git /kaggle/working/satdiff-v2
!mv /kaggle/working/ckpt_backup /kaggle/working/satdiff-v2/checkpoints
!ls -la /kaggle/working/satdiff-v2/checkpoints/

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
mv: cannot move '/kaggle/working/ckpt_backup' to '/kaggle/working/satdiff-v2/checkpoints': No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
ls: cannot access '/kaggle/working/satdiff-v2/checkpoints/': No such file or directory


In [13]:
# 2. Code, deps, HF auth. The repo is public, so no token is needed to clone.
import os, sys

REPO = '/kaggle/working/satdiff-v2'

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded')
except Exception as e:
    print(f'WARNING: no HF_TOKEN ({e}). Training still runs, but checkpoints')
    print('will not be pushed anywhere durable — a dead session loses the run.')

%cd /kaggle/working
if not os.path.isdir(REPO):
    !git clone -q https://github.com/DrKingSchultz69/satdiff-v2.git
%cd $REPO
!git pull -q

!pip install -q -r requirements.txt

os.environ['PYTHONPATH'] = f'{REPO}/src'
sys.path.insert(0, f'{REPO}/src')
print('ready')

HF_TOKEN loaded
/kaggle/working
/kaggle/working/satdiff-v2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 3.1 MB/s eta 0:00:00
ready


In [14]:
# 3. Data — 94 MB, ~2 min. Splits are assigned by SHA-256 of each file path,
# not listdir() order, so this reproduces the exact same train/val/test split
# as every other machine. That is what makes resuming here sound.
%cd /kaggle/working/satdiff-v2
!python scripts/download_data.py
!python scripts/make_splits.py

/kaggle/working/satdiff-v2
    9%     8.5 MB
   10%     9.4 MB
   20%    18.9 MB
   30%    28.3 MB
   40%    37.7 MB
   50%    47.1 MB
   60%    56.6 MB
   70%    66.0 MB
   80%    75.4 MB
   90%    84.9 MB
  100%    94.3 MB

Extracting...
Copying data/_staging/extracted/2750 -> data/v1_eurosat_rgb_64/images

Classes (10): ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Images: 27000

Done. Next: python scripts/make_splits.py
class                     train    val   test
AnnualCrop                 2404    288    308
Forest                     2382    296    322
HerbaceousVegetation       2370    294    336
Highway                    1992    269    239
Industrial                 2014    240    246
Pasture                    1616    193    191
PermanentCrop              1971    264    265
Residential                2404    291    305
River                      1994    256    250
SeaLake              

In [15]:
# 4. Train. --resume pulls last.pt from the Hub when it is not on disk, so a
# killed session costs at most `hub_push_every` epochs.
#
# CHECK THE FIRST LINES. 'resumed from epoch N' means it worked. 'starting
# fresh' means the Hub had nothing and you are about to redo every GPU-hour
# already spent — stop it and fix the checkpoint before letting it run.
%cd /kaggle/working/satdiff-v2
!PYTHONPATH=src python -m satdiff.train --config configs/v1.yaml --resume

/kaggle/working/satdiff-v2
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
checkpoints -> /kaggle/working/satdiff-v2/checkpoints/last.pt
grids       -> /kaggle/working/satdiff-v2/results/grids
device=cuda  config=e12ef601
train batches/epoch: 336
resumed from epoch 45
epoch 47/100: 100%|██████████████| 336/336 [06:06<00:00,  1.09s/it, loss=0.0209]
epoch 47  loss 0.0232  367s  ~5.4h remaining
epoch 48/100: 100%|██████████████| 336/336 [06:22<00:00,  1.14s/it, loss=0.0286]
epoch 48  loss 0.0238  382s  ~5.5h remaining
epoch 49/100: 100%|██████████████| 336/336 [06:21<00:00,  1.14s/it, loss=0.0256]
epoch 49  loss 0.0231  382s  ~5.4h remaining
epoch 50/100: 100%|██████████████| 336/336 [06:22<00:00,  1.14s/it, loss=0.0219]
epoch 50  loss 0.0235 

In [4]:
import os
from huggingface_hub import HfApi
tok = os.environ.get('HF_TOKEN', '')
print('starts:', repr(tok[:6]), 'length:', len(tok))
print(HfApi().whoami(token=tok))

starts: '' length: 0


LocalProtocolError: Illegal header value b'Bearer '

In [1]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo('DrKingSchultz69/satdiff-v1', repo_type='model',
                private=True, exist_ok=True)
api.upload_file(path_or_fileobj='/kaggle/working/satdiff-v2/checkpoints/last.pt',
                path_in_repo='last.pt',
                repo_id='DrKingSchultz69/satdiff-v1', repo_type='model')
print('uploaded')

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a782277-29c86d6471a71bc76f49f476;6b70ba22-9c05-4c32-9b4d-b5bc8d3fd4fb)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

In [ ]:
# 5. The eye test. One row per class, same 4 seeds every time.
# Four identical images in a row is mode collapse, whatever KID says.
import glob
from IPython.display import Image, display

grids = sorted(glob.glob('/kaggle/working/satdiff-v2/results/grids/*.png'))
if grids:
    print(grids[-1])
    display(Image(grids[-1]))
else:
    print('none yet — the first grid lands at epoch 5')

In [ ]:
# 6. Eval: KID + CAS. Trains a ResNet-18 on real data first (~10 min), then
# scores 2,700 generated images. ~30 min total.
#
# Bars, fixed before training started (docs/eval-plan.md):
#   KID  ship <0.05   good <0.02
#   CAS  ship >=65%   good >=80%
%cd /kaggle/working/satdiff-v2
!PYTHONPATH=src python -m satdiff.eval --config configs/v1.yaml --split val

In [ ]:
# 7. Every eval run so far, newest last.
import os
import pandas as pd

csv = '/kaggle/working/satdiff-v2/results/experiments.csv'
display(pd.read_csv(csv)) if os.path.exists(csv) else print('no eval runs yet')